# Lesson 15: Characters vs word pieces: a fair comparison

**Why:** we now have two GPTs trained **exactly the same way** (same size, same 3,000 steps, same data, about 40 minutes each). The only difference is how they read text:

| | Character model (Lesson 12) | BPE model (this lesson) |
|---|---|---|
| one token = | 1 character | a word piece (2.2 characters on average) |
| vocabulary | 85 | 512 |
| 128 tokens of context = | 128 characters | about 280 characters |

**The trap:** their loss numbers **can't be compared directly**. Choosing 1 of 512 word pieces is a harder question than choosing 1 of 85 characters, but each answer covers more text. It's like comparing a price per item with a price per kilo.

**The fix: bits per character (BPC).** We count the model's total "surprise" over the **same validation text**, and divide by the number of **characters**, whichever way it was cut into tokens:

$$\text{BPC} = \frac{\text{total loss over the text}}{\text{number of characters} \times \ln 2}$$

(Dividing by ln 2 converts the loss into **bits**: roughly, how many yes/no questions the model needs per character. Lower is better.)

## 1. Load both models

In [ ]:
import math
import re
import time
import torch
import matplotlib.pyplot as plt

from tinygpt.data import get_batch, load_text
from tinygpt.train import CHECKPOINT_DIR, load_model

text = load_text()
val_text = text[int(0.9 * len(text)):]       # the last 10%: neither model trained on it

models = {}
for name, file in [("character", "tesla_best.pt"), ("BPE", "tesla_bpe_best.pt")]:
    model, tok, ckpt = load_model(CHECKPOINT_DIR / file)
    models[name] = {"model": model, "tok": tok, "ckpt": ckpt}
    print(f"{name:10} vocab {model.config.vocab_size:4d} | {model.num_params():,} parameters | "
          f"val loss {ckpt['best_val']:.3f} per token | {ckpt['minutes']:.0f} min of training")

The BPE model has a few more parameters: its token table and output layer have 512 rows instead of 85. Its loss *per token* looks much **worse**. Now let's compare them fairly.

## 2. Bits per character

**Predict:** measured per character on the same text, which model will win: the **character** model, the **BPE** model, or a tie?

In [ ]:
@torch.no_grad()
def bits_per_char(model, tok, text, batches=100):
    ids = torch.tensor(tok.encode(text))
    chars_per_token = len(text) / len(ids)
    torch.manual_seed(0)
    loss = sum(model(*get_batch(ids, 32, model.config.block_size))[1].item() for _ in range(batches)) / batches
    return loss / chars_per_token / math.log(2), chars_per_token

for name, m in models.items():
    m["bpc"], m["cpt"] = bits_per_char(m["model"], m["tok"], val_text)
    print(f"{name:10} {m['cpt']:.2f} characters per token  ->  {m['bpc']:.3f} bits per character")

## 3. Learning over time, on the fair scale

Both training histories converted to bits per character, plotted against **minutes of training** (what actually costs you).

In [ ]:
plt.figure(figsize=(9, 4))
for name, m in models.items():
    h = m["ckpt"]["history"]
    plt.plot([r["minutes"] for r in h], [r["val"] / m["cpt"] / math.log(2) for r in h], "o-", label=name)
plt.xlabel("minutes of training"); plt.ylabel("validation bits per character")
plt.ylim(1.0, 4.0); plt.legend(); plt.title("Characters vs word pieces"); plt.show()

## 4. Their writing, side by side

Same prompt, same temperature (0.8), same random seed, about 600 characters each. We also measure the share of **real words**: generated words that actually appear somewhere in Tesla's book.

In [ ]:
book_words = set(re.findall(r"[a-z]+", text.lower()))
prompt = "The alternating current "

for name, m in models.items():
    tok, model = m["tok"], m["model"]
    n_tokens = round(600 / m["cpt"])                  # about 600 characters for both
    torch.manual_seed(42)
    start = time.time()
    out = tok.decode(model.generate(torch.tensor([tok.encode(prompt)]), n_tokens, temperature=0.8)[0].tolist())
    seconds = time.time() - start
    words = re.findall(r"[a-z]+", out.lower())
    m["real"] = sum(w in book_words for w in words) / len(words)
    print(f"===== {name} model: {m['real']:.0%} real words, written in {seconds:.1f}s =====")
    print(out)
    print()

## 5. The whole course on one scale

In [ ]:
LN2 = math.log(2)
journey = [
    ("know nothing (Lesson 3)", 4.443 / LN2),
    ("character frequencies (Lesson 4)", 3.046 / LN2),
    ("bigram (Lesson 6)", 2.406 / LN2),
    ("one attention head (Lesson 8)", 2.31 / LN2),
    ("4 heads + feed-forward (Lesson 9)", 1.90 / LN2),
    ("first GPT, 6.5 min (Lesson 10)", 1.51 / LN2),
    ("character GPT, 37 min (Lesson 12)", models["character"]["bpc"]),
    ("BPE GPT, same budget (Lesson 15)", models["BPE"]["bpc"]),
]
for name, bpc in journey:
    print(f"{name:36} {bpc:5.2f} bits/char  " + "#" * round(bpc * 8))

## 6. Final check

In [ ]:
assert all(0.5 < m["bpc"] < 3 for m in models.values())
print("Lesson 15 complete.")

## Wrap-up question

Which tokenizer would you pick for the final Tesla model, and why? Think about the score, the quality of the writing, speed, and what each one can handle (remember ć).